In [ ]:
!pip install -q \
haystack-ai \
pypdfium2 \
pillow \
sentence-transformers \
transformers \
torch \
pypdf \
nltk

## 📂 Important Step: Upload the required files

Before running the code cells, upload the two files used throughout this notebook:

👉 `2408.09869v5.pdf` — the sample PDF document used for document extraction, metadata extraction, and the RAG pipeline.

👉 `file-7mNhEGHpd0.png` — the sample image used for image content extraction and metadata extraction.

---

### 🔼 How to upload in Google Colab

1. In the **left sidebar**, click the 📁 **Files** tab.
2. Click the **Upload** button.
3. Select **`2408.09869v5.pdf`** and **`file-7mNhEGHpd0.png`** from your computer.
4. Both files should appear under **`/content/`**.

---

### ⚠️ Notes

- Keep the filenames exactly as:
  - `/content/2408.09869v5.pdf`
  - `/content/file-7mNhEGHpd0.png`
- The notebook references these filenames directly. If you rename either file, update the corresponding file paths in the code.
- If you restart the Colab runtime, you'll need to upload both files again before executing the notebook.

### 🔑 Environment Configuration

Before initializing the model, make sure to **set the following environment variables** with your correct Azure values:  

```python
os.environ['AZURE_OPENAI_API_KEY'] = "<your-api-key>"
os.environ['AZURE_OPENAI_ENDPOINT'] = "<your-endpoint-url>"
os.environ['AZURE_OPENAI_API_VERSION'] = "<supported-api-version>"
os.environ['AZURE_OPENAI_LLM_DEPLOYMENT'] = "<your-deployment-name>"


In [ ]:
from haystack.utils import Secret
from haystack.components.generators.chat import AzureOpenAIChatGenerator
import os

os.environ["AZURE_OPENAI_API_KEY"] = "FkrroqazToY6a2bUq6XJ3w3AAABACOGKkmt"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https:ai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"] = "202preview"
os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"] = "g5"



# Convenience aliases used by a later cell
GPT_DEPLOYMENT_NAME    = os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"]
GPT_AZURE_ENDPOINT     = os.environ["AZURE_OPENAI_ENDPOINT"]
GPT_OPENAI_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
GPT_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]


chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    generation_kwargs={"reasoning_effort": "minimal"}
)

## Tester

### Chat Test

In [ ]:
import os

from haystack.utils import Secret
from haystack.dataclasses import ChatMessage
from haystack.components.generators.chat import AzureOpenAIChatGenerator


result = chat_generator.run(
    messages=[
        ChatMessage.from_user(
            "Reply with exactly: Azure GPT is working."
        )
    ]
)

print(result["replies"][0].text)

Azure GPT is working.


### Vision Test

In [ ]:
import os
import base64
from openai import AzureOpenAI

# Read the image
IMAGE_PATH = "/content/file-7mNhEGHpd0.png"

with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

# Azure OpenAI client
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

# Send image to GPT
response = client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Extract all text from this image exactly as it appears. Preserve the reading order."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_b64}"
                    }
                }
            ]
        }
    ],
    max_completion_tokens=32768,
)

print(response.choices[0].message.content)

United Healthcare (87726)
10 Main Street
Madison AL 123456

HEALTH INSURANCE CLAIM FORM
APPROVED BY NATIONAL UNIFORM CLAIM COMMITTEE (NUCC) 02/12

PICA

1. MEDICARE (Medicare)  MEDICAID (Medicaid)  TRICARE  CHAMPVA (Member ID#)  GROUP HEALTH PLAN (ID#)  FECA BLK LUNG (ID#)  Other (ID#)
1a. INSURED'S I.D. NUMBER (For Program in Item 1)
1235678910
PICA

2. PATIENT'S NAME (Last Name, First Name, Middle Initial)
Jones, Mariska

3. PATIENT'S BIRTH DATE
MM  DD  YY
01  01  00
SEX
M  F

4. INSURED'S NAME (Last Name, First Name, Middle Initial)
Jones, Mariska

5. PATIENT'S ADDRESS (No., Street)
967 Elm Street
CITY
Austin
STATE
TX
ZIP CODE
73301
TELEPHONE (Include Area Code)
( 845 ) 3130545

6. PATIENT RELATIONSHIP TO INSURED
Self  Spouse  Child  Other

7. INSURED'S ADDRESS (No., Street)
10 Main Street
CITY
Madison
STATE
AL
ZIP CODE
123456
TELEPHONE (Include Area Code)
( 212 ) 6461234

8. RESERVED FOR NUCC USE

9. OTHER INSURED'S NAME (Last Name, First Name, Middle Initial)
Jones, Jonathan
a. OT

## 📄 Extracting Content from a PDF using `LLMDocumentContentExtractor`

In this example, we use Haystack's **`LLMDocumentContentExtractor`** to extract text from a PDF document using the shared Azure OpenAI `chat_generator`.

Instead of relying on traditional PDF parsing, the document is analyzed by a multimodal Large Language Model (LLM), which understands both the text and the layout of the page.

### 🔍 What this code does

- Creates an `LLMDocumentContentExtractor`.
- Loads the PDF as a Haystack `Document`.
- Sends the document to the Azure OpenAI model.
- Receives the extracted text.
- Prints the extracted content.

---

### 🔄 Extraction Workflow

```text
PDF Document
      │
      ▼
Haystack Document
      │
      ▼
LLMDocumentContentExtractor
      │
      ▼
Azure OpenAI
      │
      ▼
Extracted Text
```

---

### 📌 Output

The extracted content is returned as a Haystack `Document` and can be used for tasks such as:

- Question Answering
- Retrieval-Augmented Generation (RAG)
- Document Summarization
- Semantic Search
- Metadata Extraction

In [ ]:
from haystack import Document
from haystack.components.extractors.image import LLMDocumentContentExtractor

# Create the extractor using the existing chat_generator
extractor = LLMDocumentContentExtractor(
    chat_generator=chat_generator,
    file_path_meta_field="file_path",
    raise_on_failure=False
)

# Create document pointing to the uploaded PDF
documents = [
    Document(
        content="",
        meta={
            "file_path": "/content/2408.09869v5.pdf",
            "page_number": 1
        }
    )
]

# Run the extractor
result = extractor.run(documents=documents)

# Summary
print(f"Successfully processed: {len(result['documents'])}")
print(f"Failed documents: {len(result['failed_documents'])}")

# Display extracted content
for doc in result["documents"]:
    print(f"File: {doc.meta['file_path']}")
    print("\nExtracted content:\n")
    print(doc.content)
    print("\n" + "=" * 80 + "\n")

Successfully processed: 1
Failed documents: 0
File: /content/2408.09869v5.pdf

Extracted content:

[img-caption]Cartoonish orange mascot holding a sheet of paper, centered at the top of the page.[/img-caption]

Docling Technical Report

Version 1.0

Christoph Auer    Maksym Lysak    Ahmed Nassar    Michele Dolfi    Nikolaos Livathinos
Panos Vagenas    Cesar Berrospi Ramis    Matteo Omenetti    Fabian Lindlbauer
Kasper Dinkla    Lokesh Mishra    Yusik Kim    Shubham Gupta    Rafael Teixeira de Lima
Valery Weber    Lucas Morin    Ingmar Meijer    Viktor Kuropiatnyk    Peter W. J. Staar

AI4K Group, IBM Research
Rüschlikon, Switzerland

Abstract

This technical report introduces Docling, an easy to use, self-contained, MIT-licensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The co

## 🖼️ Extracting Text from an Image using a Custom Prompt

In this example, we use **`LLMDocumentContentExtractor`** to extract text from an image document. Unlike the previous example, we provide a **custom prompt** that tells the LLM how the extracted content should be formatted.

### What this code does

- Creates a custom prompt with instructions for the LLM.
- Initializes an `LLMDocumentContentExtractor` using the shared `chat_generator`.
- Loads the image (`file-7mNhEGHpd0.png`) as a Haystack `Document`.
- Sends the image and prompt to the Azure OpenAI model.
- Prints the extracted text returned by the model.

### Extraction Flow

```text
Image
   │
   ▼
LLMDocumentContentExtractor
   │
   ▼
Azure OpenAI
   │
   ▼
Extracted Text
```

### Why use a Custom Prompt?

A custom prompt allows you to control how the LLM extracts information. For example, you can instruct it to:

- Preserve the reading order.
- Format tables as Markdown.
- Describe diagrams or images.
- Maintain the document structure.

This makes the extractor flexible for different document types such as invoices, insurance forms, receipts, or scanned documents.

In [ ]:
from haystack import Document
from haystack.components.extractors.image import LLMDocumentContentExtractor

# Custom prompt
custom_prompt = """
Extract all text content from this image-based document.

Instructions:
- Extract text exactly as it appears
- Preserve the reading order
- Format tables as markdown
- Describe any images or diagrams briefly
- Maintain document structure

Document:
"""

# Create the extractor using the shared chat_generator
extractor = LLMDocumentContentExtractor(
    chat_generator=chat_generator,
    prompt=custom_prompt,
    file_path_meta_field="file_path",
    raise_on_failure=False
)

# Create document
documents = [
    Document(
        content="",
        meta={
            "file_path": "/content/file-7mNhEGHpd0.png"
        }
    )
]

# Run the extractor
result = extractor.run(documents=documents)

# Print results
for doc in result["documents"]:
    print(f"File: {doc.meta['file_path']} (Page {doc.meta.get('page_number', 'N/A')})")
    print("\nExtracted content:\n")
    print(doc.content)
    print("\n" + "=" * 80 + "\n")

File: /content/file-7mNhEGHpd0.png (Page N/A)

Extracted content:

Document title: HEALTH INSURANCE CLAIM FORM
Approved by National Uniform Claim Committee (NUCC) 02/12

[Small QR code image in the upper-left corner]

PICA

1. MEDICARE (Medicare)
2. MEDICAID (Medicaid)
3. TRICARE (CHAMPUS)
4. CHAMPVA
5. GROUP HEALTH PLAN
6. FECA BLK LUNG
7. OTHER

1a. INSURED’S I.D. NUMBER (For Program in Item 1)
1235678910

2. PATIENT’S NAME (Last Name, First Name, Middle Initial)
Jones, Mariska

3. PATIENT’S BIRTH DATE MM DD YY
01 01 00
SEX M F X

4. INSURED’S NAME (Last Name, First Name, Middle Initial)
Jones, Mariska

5. PATIENT’S ADDRESS (No., Street)
967 Elm Street

6. PATIENT RELATIONSHIP TO INSURED
Self [X]  Spouse [ ]  Child [ ]  Other [ ]

7. INSURED’S ADDRESS (No., Street)
10 Main Street

CITY
Austin

STATE
TX

8. RESERVED FOR NUCC USE

CITY
Madison

STATE
AL

ZIP CODE
73301

TELEPHONE (Include Area Code)
(845) 3130545

ZIP CODE
123456

TELEPHONE (Include Area Code)
(212) 6461234

9. OTHER I

## 🔄 Building a Haystack Pipeline for Document Processing

In this example, we create a simple **Haystack Pipeline** that processes documents in three steps:

1. **Extract** the content from the document using an LLM.
2. **Split** the extracted content into smaller chunks.
3. **Store** the processed documents in an in-memory document store.

Pipelines allow multiple Haystack components to work together by passing the output of one component as the input to the next.

### 🔍 What this code does

- Creates an `InMemoryDocumentStore` to store processed documents.
- Builds a pipeline with three components:
  - `LLMDocumentContentExtractor`
  - `DocumentSplitter`
  - `DocumentWriter`
- Connects the components to define the processing flow.
- Processes both a PDF and an image.
- Stores the processed documents in memory.
- Prints the number of processed documents and verifies that they have been stored successfully.

---

### 🔄 Pipeline Workflow

```text
PDF / Image
      │
      ▼
LLMDocumentContentExtractor
      │
      ▼
DocumentSplitter
      │
      ▼
DocumentWriter
      │
      ▼
InMemoryDocumentStore
```

---

### 📌 Pipeline Components

- **LLMDocumentContentExtractor** – Extracts text from the document using the configured Azure OpenAI model.
- **DocumentSplitter** – Splits the extracted content into smaller chunks, making it easier for downstream tasks such as Retrieval-Augmented Generation (RAG).
- **DocumentWriter** – Stores the processed documents in the `InMemoryDocumentStore`.

---

### 📌 Why use a Pipeline?

A pipeline provides a structured way to chain multiple processing steps together. Instead of running each component manually, Haystack automatically passes the output of one component to the next, making document processing more modular, reusable, and easier to maintain.

In [ ]:
from haystack import Pipeline
from haystack.components.extractors.image import LLMDocumentContentExtractor
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document

# Create document store
document_store = InMemoryDocumentStore()

# Create pipeline
p = Pipeline()

p.add_component(
    instance=LLMDocumentContentExtractor(
        chat_generator=chat_generator,
        file_path_meta_field="file_path"
    ),
    name="content_extractor"
)

p.add_component(
    instance=DocumentSplitter(),
    name="splitter"
)

p.add_component(
    instance=DocumentWriter(document_store=document_store),
    name="writer"
)

# Connect components
p.connect("content_extractor.documents", "splitter.documents")
p.connect("splitter.documents", "writer.documents")

# Create test documents
docs = [
    Document(
        content="",
        meta={
            "file_path": "/content/2408.09869v5.pdf",
            "page_number": 1
        }
    ),
    Document(
        content="",
        meta={
            "file_path": "/content/file-7mNhEGHpd0.png"
        }
    )
]

# Run pipeline
# ADDED: Return outputs from the intermediate content_extractor component
result = p.run(
    {
        "content_extractor": {
            "documents": docs
        }
    },
    include_outputs_from={"content_extractor"}
)

# Check results
processed_docs = result.get("content_extractor", {}).get("documents", [])
failed_docs = result.get("content_extractor", {}).get("failed_documents", [])

print(f"Successfully processed: {len(processed_docs)}")
print(f"Failed documents: {len(failed_docs)}")

# Access documents in the store
stored_docs = document_store.filter_documents()

print(f"Documents in store: {len(stored_docs)}")

Successfully processed: 2
Failed documents: 0
Documents in store: 4


In [ ]:
for doc in stored_docs:
    print(f"File: {doc.meta.get('file_path')}")
    print("Extracted content:\n")
    print(doc.content)
    print("\n" + "="*80 + "\n")

File: /content/2408.09869v5.pdf
Extracted content:

# [img-caption]An orange cartoon owl holding a sheet of paper, centered near the top of the page.[/img-caption]

Docling Technical Report

Version 1.0

Christoph Auer    Maksym Lysak    Ahmed Nassar    Michele Dolfi    Nikolaos Livathinos
Panos Vagenas    Cesar Berrospi Ramis    Matteo Omenetti    Fabian Lindlbauer
Kasper Dinkla    Lokesh Mishra    Yusik Kim    Shubham Gupta    Rafael Teixeira de Lima
Valery Weber    Lucas Morin    Ingmar Meijer    Viktor Kuropiatnyk    Peter W. J. Staar

AI4K Group, IBM Research
Rüschlikon, Switzerland

Abstract

This technical report introduces Docling, an easy to use, self-contained, MIT-licensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and

## 🏷️ Extracting Metadata using `LLMMetadataExtractor`

In this example, we use **`LLMMetadataExtractor`** to extract structured metadata from text using a custom prompt. Here, the prompt performs **Named Entity Recognition (NER)** by identifying entities such as companies, organizations, people, countries, products, and services.

### 🔍 What this code does

- Creates sample text documents.
- Defines a custom prompt for Named Entity Recognition (NER).
- Initializes an `LLMMetadataExtractor` using the shared `chat_generator`.
- Sends each document to the configured GPT model.
- Stores the extracted entities as metadata in each Haystack `Document`.
- Prints the extracted metadata.

---

### 🔄 Metadata Extraction Workflow

```text
Document Text
      │
      ▼
LLMMetadataExtractor
      │
      ▼
Configured GPT Model
      │
      ▼
Extracted Metadata
      │
      ▼
Haystack Document
(with metadata)
```

---

### 📌 Why use `LLMMetadataExtractor`?

`LLMMetadataExtractor` allows you to enrich documents with structured information extracted by an LLM. By simply changing the prompt, you can extract different types of metadata such as:

- Named Entities (NER)
- Keywords
- Topics
- Categories
- Tags
- Custom fields

The extracted metadata can later be used for filtering, searching, indexing, or improving Retrieval-Augmented Generation (RAG) pipelines.

In [ ]:
from haystack import Document
from haystack.components.extractors.llm_metadata_extractor import LLMMetadataExtractor

# Example documents
docs = [
    Document(
        content="deepset was founded in 2018 in Berlin, and is known for its Haystack framework"
    ),
    Document(
        content="Hugging Face is a company founded in New York, USA and is known for its Transformers library"
    )
]

# Custom NER Prompt
NER_PROMPT = """
-Goal-
Given text and a list of entity types, identify all entities of those types from the text.

-Steps-
1. Identify all entities. For each identified entity, extract:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types:
  [organization, product, service, industry]

Format each entity as:
{"entity": <entity_name>, "entity_type": <entity_type>}

2. Return all entities in a single JSON list.

-Examples-
######################
Example 1:
entity_types: [organization, person, partnership, financial metric, product, service, industry, investment strategy, market trend]
text: Another area of strength is our co-brand issuance. Visa is the primary network partner for eight of the top
10 co-brand partnerships in the US today and we are pleased that Visa has finalized a multi-year extension of
our successful credit co-branded partnership with Alaska Airlines, a portfolio that benefits from a loyal customer
base and high cross-border usage.
We have also had significant co-brand momentum in CEMEA. First, we launched a new co-brand card in partnership
with Qatar Airways, British Airways and the National Bank of Kuwait. Second, we expanded our strong global
Marriott relationship to launch Qatar's first hospitality co-branded card with Qatar Islamic Bank. Across the
United Arab Emirates, we now have exclusive agreements with all the leading airlines marked by a recent
agreement with Emirates Skywards.
And we also signed an inaugural Airline co-brand agreement in Morocco with Royal Air Maroc.

output:
{
  "entities": [
    {"entity": "Visa", "entity_type": "company"},
    {"entity": "Alaska Airlines", "entity_type": "company"},
    {"entity": "Qatar Airways", "entity_type": "company"},
    {"entity": "British Airways", "entity_type": "company"},
    {"entity": "National Bank of Kuwait", "entity_type": "company"},
    {"entity": "Marriott", "entity_type": "company"},
    {"entity": "Qatar Islamic Bank", "entity_type": "company"},
    {"entity": "Emirates Skywards", "entity_type": "company"},
    {"entity": "Royal Air Maroc", "entity_type": "company"}
  ]
}

#############################

-Real Data-
######################
entity_types: [company, organization, person, country, product, service]
text: {{ document.content }}

######################
output:
"""

# Initialize the metadata extractor using the shared chat_generator
extractor = LLMMetadataExtractor(
    prompt=NER_PROMPT,
    chat_generator=chat_generator,
    expected_keys=["entities"],
    raise_on_failure=False,
)

# Warm up the extractor
extractor.warm_up()

# Run extraction
result = extractor.run(documents=docs)

# Display results
for doc in result["documents"]:
    print(f"Text: {doc.content}")
    print(f"Extracted metadata: {doc.meta}")
    print("\n" + "=" * 60 + "\n")

Text: deepset was founded in 2018 in Berlin, and is known for its Haystack framework
Extracted metadata: {'entities': [{'entity': 'Deepset', 'entity_type': 'organization'}, {'entity': 'Berlin', 'entity_type': 'country'}, {'entity': 'Haystack', 'entity_type': 'product'}]}


Text: Hugging Face is a company founded in New York, USA and is known for its Transformers library
Extracted metadata: {'entities': [{'entity': 'Hugging Face', 'entity_type': 'company'}, {'entity': 'New York', 'entity_type': 'country'}, {'entity': 'USA', 'entity_type': 'country'}, {'entity': 'Transformers', 'entity_type': 'product'}]}




## 🤖 Building a Retrieval-Augmented Generation (RAG) Pipeline

In this example, we build an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline using Haystack. The pipeline ingests a PDF document, converts it into searchable document chunks, stores their vector embeddings in a document store, retrieves the most relevant information for a user query, and uses an LLM to generate a context-aware response.

### 🔍 What this code does

- Loads a PDF document.
- Converts the PDF into Haystack `Document` objects using `PyPDFToDocument`.
- Splits the extracted text into smaller chunks using `DocumentSplitter`.
- Generates vector embeddings for each chunk using `SentenceTransformersDocumentEmbedder`.
- Stores the embedded documents in an `InMemoryDocumentStore`.
- Converts the user's question into an embedding using `SentenceTransformersTextEmbedder`.
- Retrieves the most relevant document chunks using `InMemoryEmbeddingRetriever`.
- Builds a prompt containing the retrieved context using `ChatPromptBuilder`.
- Sends the prompt to the configured GPT model to generate the final answer.

---

### 🔄 RAG Workflow

```text
                    PDF Document
                         │
                         ▼
                 PyPDFToDocument
                         │
                         ▼
               DocumentSplitter
                         │
                         ▼
 SentenceTransformersDocumentEmbedder
                         │
                         ▼
              DocumentWriter
                         │
                         ▼
         InMemoryDocumentStore
                         ▲
                         │
                  User Question
                         │
                         ▼
   SentenceTransformersTextEmbedder
                         │
                         ▼
      InMemoryEmbeddingRetriever
                         │
                         ▼
             Retrieved Chunks
                         │
                         ▼
             ChatPromptBuilder
                         │
                         ▼
          Configured GPT Model
                         │
                         ▼
                 Generated Answer
```

---

### 📌 Why use a RAG Pipeline?

A Retrieval-Augmented Generation (RAG) pipeline enables an LLM to answer questions using information from external documents instead of relying solely on its pretrained knowledge. The retrieval step supplies relevant document chunks as context, allowing the LLM to generate responses that are more accurate, grounded, and specific to the provided documents.

A typical RAG pipeline consists of:

- Document ingestion
- Document conversion
- Text chunking
- Embedding generation
- Vector storage
- Semantic retrieval
- Prompt construction
- LLM-based answer generation using retrieved context

This architecture is widely used for document question answering, enterprise search, knowledge assistants, and chatbots that need to answer questions based on custom documents.

In [ ]:
from haystack import Pipeline
from haystack.dataclasses import Document, ChatMessage

from haystack.document_stores.in_memory import InMemoryDocumentStore

from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentSplitter

from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)

from haystack.components.writers import DocumentWriter
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import AzureOpenAIChatGenerator
from haystack.utils import Secret
import os

In [ ]:
# Create document store
document_store = InMemoryDocumentStore()

In [ ]:
indexing_pipeline = Pipeline()

indexing_pipeline.add_component(
    "converter",
    PyPDFToDocument()
)

indexing_pipeline.add_component(
    "splitter",
    DocumentSplitter(
        split_by="word",
        split_length=200,
        split_overlap=50
    )
)

indexing_pipeline.add_component(
    "embedder",
    SentenceTransformersDocumentEmbedder(
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
)

indexing_pipeline.add_component(
    "writer",
    DocumentWriter(document_store=document_store)
)

indexing_pipeline.connect(
    "converter.documents",
    "splitter.documents"
)

indexing_pipeline.connect(
    "splitter.documents",
    "embedder.documents"
)

indexing_pipeline.connect(
    "embedder.documents",
    "writer.documents"
)

🚅 Components
  - converter: PyPDFToDocument
  - splitter: DocumentSplitter
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - converter.documents -> splitter.documents (list[Document])
  - splitter.documents -> embedder.documents (list[Document])
  - embedder.documents -> writer.documents (list[Document])

In [ ]:
from haystack.dataclasses import ByteStream

with open("/content/2408.09869v5.pdf", "rb") as f:
    pdf_bytes = f.read()

indexing_pipeline.run(
    {
        "converter": {
            "sources": [
                ByteStream(
                    data=pdf_bytes,
                    mime_type="application/pdf"
                )
            ]
        }
    }
)

print("PDF indexed successfully.")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

PDF indexed successfully.


In [ ]:
# Prompt template
prompt_template = [
    ChatMessage.from_system(
        """
You are a helpful assistant answering questions about a PDF document.

Use ONLY the provided context to answer the user's question.

If multiple retrieved passages are relevant, combine the information into a concise answer.

If the answer is partially available, answer using the available information instead of saying it is unavailable.

Only say "I couldn't find the answer in the provided context." if none of the retrieved passages contain relevant information.

Context:

{% for doc in documents %}
{{ doc.content }}

{% endfor %}
"""
    ),
    ChatMessage.from_user(
        "{{question}}"
    )
]

In [ ]:
# RAG Pipeline
rag_pipeline = Pipeline()



rag_chat_generator = AzureOpenAIChatGenerator(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    generation_kwargs={"reasoning_effort": "minimal"}
)

rag_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(
        model="sentence-transformers/all-MiniLM-L6-v2"
    )
)

rag_pipeline.add_component(
    "retriever",
    InMemoryEmbeddingRetriever(
        document_store=document_store,
        top_k=10
    )
)

rag_pipeline.add_component(
    "prompt_builder",
    ChatPromptBuilder(template=prompt_template)
)

rag_pipeline.add_component(
    "llm",
    rag_chat_generator
)

rag_pipeline.connect(
    "text_embedder.embedding",
    "retriever.query_embedding"
)

rag_pipeline.connect(
    "retriever.documents",
    "prompt_builder.documents"
)

rag_pipeline.connect(
    "prompt_builder.prompt",
    "llm.messages"
)

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: AzureOpenAIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [ ]:
# Ask a Question
question = "What is docling."

result = rag_pipeline.run(
    {
        "text_embedder": {
            "text": question
        },
        "prompt_builder": {
            "question": question
        }
    }
)

print(result["llm"]["replies"][0].text)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Docling is an open-source, MIT-licensed Python package for converting PDF documents into machine-processable formats like JSON or Markdown. It runs locally on commodity hardware and uses specialized AI models for document layout analysis (trained on DocLayNet) and table structure recognition (TableFormer). The pipeline parses PDFs to extract text tokens and page images, applies page-level AI models to detect layout and tables, optionally performs OCR for scanned content, and then assembles results with metadata extraction (title, authors, references, language), reading-order correction, and figure–caption matching. It offers configurable features (e.g., OCR, table recognition), runtime options (thread budget, backends), and outputs suitable for applications like knowledge-base construction and RAG.
